In [0]:
transactions_df = spark.read.table("silver.transactions")
users_df = spark.read.table("silver.users")
cards_df = spark.read.table("silver.cards")


In [0]:
#Which day(s) of the week sees the highest number of fraudulent transactions?
from pyspark.sql.functions import col, count, when

gold_fraud_by_day = (
    transactions_df
    .filter(col("is_fraud") == True)
    .groupBy("day_of_week")
    .agg(count("*").alias("fraud_count"))
    .withColumn(
        "day_name",
        when(col("day_of_week") == 1, "Sunday")
        .when(col("day_of_week") == 2, "Monday")
        .when(col("day_of_week") == 3, "Tuesday")
        .when(col("day_of_week") == 4, "Wednesday")
        .when(col("day_of_week") == 5, "Thursday")
        .when(col("day_of_week") == 6, "Friday")
        .when(col("day_of_week") == 7, "Saturday")
    )
    .orderBy(col("fraud_count").desc())
)

display(gold_fraud_by_day)

day_of_week,fraud_count,day_name
1,2646,Sunday
6,2284,Friday
5,2082,Thursday
3,2037,Tuesday
2,1747,Monday
7,1434,Saturday
4,1102,Wednesday


Databricks visualization. Run in Databricks to view.

In [0]:
#What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?
from pyspark.sql.functions import col, max, add_months, lit, dayofmonth

last_day = transactions_df.agg(max(col('date'))).collect()[0][0]
total_transactions_daily = (
    transactions_df
    .filter(col("date") >= add_months(lit(last_day), -1))
    .groupBy(dayofmonth(col("date")))
    .agg(count("*").alias("total_count"))
)
fraud_transactions_daily = (
    transactions_df
    .filter(col("is_fraud") == True)
    .filter(col("date") >= add_months(lit(last_day), -1))
    .groupBy(dayofmonth(col("date")))
    .agg(count("*").alias("fraud_count"))
)

gold_fraud_rate = (
    total_transactions_daily
    .join(fraud_transactions_daily, on="dayofmonth(date)", how="left")
    .fillna(0)
    .withColumn("fraud_rate", col("fraud_count") / col("total_count"))
)

display(gold_fraud_rate)

dayofmonth(date),total_count,fraud_count,fraud_rate
31,3978,0,0.0
28,3944,0,0.0
26,3813,6,0.0015735641227380016
27,3866,1,2.586652871184687E-4
12,3906,13,0.0033282130056323605
22,3487,11,0.0031545741324921135
1,3363,8,0.0023788284269997025
13,3879,0,0.0
6,3973,0,0.0
16,3496,0,0.0


Databricks visualization. Run in Databricks to view.

In [0]:
#Which users have the largest number of flagged (is_fraud = true) transactions?
from pyspark.sql.functions import col, count, max

fraud_per_client = (
    transactions_df
    .filter(col("is_fraud") == True)
    .groupBy("client_id")
    .agg(count("*").alias("fraud_count"))
)

users_flagged = (
    fraud_per_client
    .join(users_df, fraud_per_client.client_id == users_df.id, "inner")
    .drop(users_df.id)
)

display(users_flagged)

display(users_flagged.filter(col("fraud_count") == fraud_per_client.agg(max(col("fraud_count"))).collect()[0][0]))


client_id,fraud_count,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,10,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1746,21,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
1718,32,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
708,8,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
68,9,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.0,41997.0,0.0,704,3
1075,7,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.0,51500.0,102286.0,672,3
1116,6,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.0,42509.0,2895.0,755,5
1752,6,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.0,38190.0,81262.0,810,1
1094,9,34,62,1985,10,Male,74786 Jefferson Drive,44.75,-85.6,20325.0,41442.0,78833.0,712,3
1660,3,41,68,1978,4,Female,40 Washington Drive,36.73,-102.51,11342.0,23123.0,5079.0,723,6


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

client_id,fraud_count,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
1102,58,49,65,1970,7,Female,836 Bayview Avenue,36.12,-95.91,46461.0,94733.0,0.0,707,5


In [0]:
#What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?
from pyspark.sql.functions import col, sum as sum, avg

weekly_spend = (
    transactions_df
    .groupBy("client_id", "week")
    .agg(sum("amount").alias("weekly_total"))
)
from pyspark.sql.functions import sum as _sum

avg_weekly_spend = (
    weekly_spend
    .groupBy("client_id")
    .agg(avg("weekly_total").alias("avg_weekly_spend"))
)

weekly_with_avg = (
    weekly_spend
    .join(avg_weekly_spend, on="client_id", how="inner")
)


spikes = (
    weekly_with_avg
    .filter(col("weekly_total") > col("avg_weekly_spend") * 1.5)
)

display(spikes)

client_id,week,weekly_total,avg_weekly_spend
1709,50,8399.8700,5101.24811321
924,8,7042.6500,4620.68886792
58,34,10296.6700,6821.91415094
1163,31,11902.1900,6436.49830189
48,6,7725.5800,4952.75415094
1242,9,9505.1400,6252.37679245
357,25,13398.7000,8877.85981132
1862,40,4020.9000,2249.95500000
1434,37,6084.1600,3468.34169811
647,43,3155.0400,2084.27000000


In [0]:
#Which merchant categories exhibit the highest fraud rate?

In [0]:
#Are there specific merchants with unusually high fraud volume?

In [0]:
#How does fraud distribution vary by time of day (morning vs night)?

In [0]:
#Whats the average transaction amount for fraud vs non-fraud transactions?

In [0]:
#What are the total monetary losses due to fraud each day?

In [0]:
#How many unique users commit fraudulent transactions per week?

In [0]:
#Do fraud patterns show seasonal or monthly spikes?

In [0]:
#How has user behavior changed before versus after a fraudulent event?

In [0]:
#Are fraudulent transactions more common on high-value purchases compared to low-value purchases?